<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>》配套补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>



# 附录C：Qwen3 LLM 源代码


本 Notebook 使用到的依赖：


In [13]:
%pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt  # 安装项目依赖
from importlib.metadata import version  # 从标准库获取已安装库版本

used_libraries = [                      # 待检查版本的库列表
    "reasoning_from_scratch",           # 项目下载函数等
    "torch",                            # 深度学习库
    "tokenizers"                        # 分词器库
]

for lib in used_libraries:              # 逐个库输出版本号
    print(f"{lib} version: {version(lib)}")


reasoning_from_scratch version: 0.1.10
torch version: 2.9.0+cu126
tokenizers version: 0.22.1


- 虽然本书强调“from scratch”，但正如前文所述，这里的 from scratch 专指推理方法，而非 LLM 本身。
- 真正从零实现一个 LLM 足以写成另一本书——这正是我另一本书 [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch) 的主题。
- 不过如果你想了解本书所用实现，本附录列出了我们通过 reasoning_from_scratch Python 包导入的 Qwen3 源码：

```python
from reasoning_from_scratch.qwen3 import Qwen3Model, Qwen3Tokenizer
```

- 请注意，这套架构代码与 [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch) 中介绍的 GPT-2 十分相似。
- 虽然本书不要求熟悉 GPT-2，但附录中仍加入了一些与 GPT-2 的对比，方便了解两者关系（这份实现正是我把 GPT-2 代码逐步迁移到 Qwen3 架构的结果）。


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F01_raschka.webp" width="500px">

&nbsp;
## C.1 均方根层归一化（RMSNorm）


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F02_raschka.webp" width="500px">

In [14]:
import torch.nn as nn                     # 导入 PyTorch 神经网络模块

class RMSNorm(nn.Module):                 # 定义 RMSNorm 层，继承自 nn.Module
    def __init__(
        self,
        emb_dim,                          # 嵌入维度
        eps=1e-6,                         # 数值稳定的 epsilon
        bias=False,                       # 是否使用可学习偏置
        qwen3_compatible=True,            # 是否按 Qwen3 兼容方式运行
    ):
        super().__init__()                # 调用父类构造
        self.eps = eps                    # 保存 epsilon
        self.qwen3_compatible = qwen3_compatible  # 保存兼容标志
        self.scale = nn.Parameter(torch.ones(emb_dim))      # 可学习缩放参数 gamma
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None  # 可学习偏置 beta（可选）

    def forward(self, x):
        input_dtype = x.dtype             # 记录输入数据类型

        if self.qwen3_compatible:         # 若需兼容 Qwen3，则先转为 float32
            x = x.to(torch.float32)

        variance = x.pow(2).mean(dim=-1, keepdim=True)  # 计算均方值
        norm_x = x * torch.rsqrt(variance + self.eps)   # RMS 归一化
        norm_x = norm_x * self.scale                    # 应用可学习缩放

        if self.shift is not None:        # 如有偏置，则加上偏置
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)     # 转回原始数据类型并返回


&nbsp;
## C.2 前馈模块


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F03_raschka.webp" width="500px">

In [15]:
class FeedForward(nn.Module):                       # 前馈全连接模块
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(                       # 第一路线性层（Gate/激活前）
            cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"],
            bias=False
        )
        self.fc2 = nn.Linear(                       # 第二路线性层（Pairwise 乘积另一支）
            cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"],
            bias=False
        )
        self.fc3 = nn.Linear(                       # 输出线性层，将隐藏维映射回嵌入维
            cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"],
            bias=False
        )

    def forward(self, x):
        x_fc1 = self.fc1(x)                         # 路径一：线性变换
        x_fc2 = self.fc2(x)                         # 路径二：线性变换
        x = nn.functional.silu(x_fc1) * x_fc2       # SiLU 激活后与另一支逐元素相乘
        return self.fc3(x)                          # 映射回嵌入维作为输出


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F04_raschka.webp" width="500px">

&nbsp;
## C.3 旋转位置嵌入（RoPE）


In [16]:
import torch  # 导入 PyTorch

def compute_rope_params(head_dim, theta_base=10_000, context_length=4096,
                        dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"  # 确保 head_dim 为偶数
    inv_freq = 1.0 / (theta_base ** (                           # 计算逆频率表 inv_freq
        torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float()
        / head_dim
    ))
    positions = torch.arange(context_length, dtype=dtype)        # 位置序列 [0, ..., context_length-1]
    angles = positions[:, None] * inv_freq[None, :]              # 广播得到位置×频率矩阵
    angles = torch.cat([angles, angles], dim=1)                  # 拼接成与 head_dim 对齐的角度矩阵

    cos = torch.cos(angles)                                      # 预计算 cos 表
    sin = torch.sin(angles)                                      # 预计算 sin 表

    return cos, sin                                              # 返回 cos/sin 查表

def apply_rope(x, cos, sin, offset=0):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape           # 解包张量形状
    assert head_dim % 2 == 0, "Head dimension must be even"      # 确保 head_dim 为偶数

    x1 = x[..., : head_dim // 2]  # 前半部分 (偶/奇维度对)
    x2 = x[..., head_dim // 2:]   # 后半部分

    # 截取对应的 cos/sin 片段并扩展维度到 (1, 1, seq_len, head_dim)
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    rotated = torch.cat((-x2, x1), dim=-1)                       # 旋转后的向量对 (-y, x)
    x_rotated = (x * cos) + (rotated * sin)                      # 应用旋转：x*cos + rot*sin

    return x_rotated.to(dtype=x.dtype)                           # 转回原始 dtype 返回


&nbsp;
## C.4 分组查询注意力（GQA）


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F05_raschka.webp" width="500px">

In [17]:
class GroupedQueryAttention(nn.Module):                             # 定义分组多头注意力（GQA）层
    def __init__(self, d_in, num_heads, num_kv_groups, head_dim=None,
                 qk_norm=False, dtype=None):
        super().__init__()                                          # 调用父类构造
        assert num_heads % num_kv_groups == 0                       # 头数必须能被 KV 组数整除

        self.num_heads = num_heads                                  # 总注意力头数
        self.num_kv_groups = num_kv_groups                          # KV 组数
        self.group_size = num_heads // num_kv_groups                # 每组包含的头数

        if head_dim is None:                                        # 若未指定 head_dim，则自动计算
            assert d_in % num_heads == 0
            head_dim = d_in // num_heads

        self.head_dim = head_dim                                    # 单头维度
        self.d_out = num_heads * head_dim                           # QKV 投影后总维度

        self.W_query = nn.Linear(                                   # 查询投影层
            d_in, self.d_out, bias=False, dtype=dtype
        )
        self.W_key = nn.Linear(                                     # 键投影层（按 KV 组输出）
            d_in, num_kv_groups * head_dim, bias=False, dtype=dtype
        )
        self.W_value = nn.Linear(                                   # 值投影层（按 KV 组输出）
            d_in, num_kv_groups * head_dim, bias=False, dtype=dtype
        )

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)  # 输出投影回输入维

        if qk_norm:                                                 # 可选 Q/K RMSNorm
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape                                  # 批大小、序列长度、特征维

        queries = self.W_query(x)                                   # 线性投影得到 Q，形状 (b, seq, H*D)
        keys = self.W_key(x)                                        # 线性投影得到 K，形状 (b, seq, G*D)
        values = self.W_value(x)                                    # 线性投影得到 V，形状 (b, seq, G*D)

        queries = queries.view(b, num_tokens, self.num_heads,       # 重排 Q -> (b, H, seq, D)
                               self.head_dim).transpose(1, 2)
        keys_new = keys.view(b, num_tokens, self.num_kv_groups,     # 重排 K -> (b, G, seq, D)
                             self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups, # 重排 V -> (b, G, seq, D)
                                 self.head_dim).transpose(1, 2)

        if self.q_norm:                                             # 如启用，对 Q 做 RMSNorm
            queries = self.q_norm(queries)
        if self.k_norm:                                             # 如启用，对 K 做 RMSNorm
            keys_new = self.k_norm(keys_new)

        queries = apply_rope(queries, cos, sin, offset=start_pos)   # 应用 RoPE 位置编码到 Q
        keys_new = apply_rope(keys_new, cos, sin, offset=start_pos) # 应用 RoPE 位置编码到 K

        if cache is not None:                                       # 若有缓存则拼接历史 K/V
            prev_k, prev_v = cache
            keys = torch.cat([prev_k, keys_new], dim=2)
            values = torch.cat([prev_v, values_new], dim=2)
        else:
            start_pos = 0                                           # 无缓存时起始位置重置
            keys, values = keys_new, values_new
        next_cache = (keys, values)                                 # 更新缓存返回

        keys = keys.repeat_interleave(self.group_size, dim=1)       # 将 K 按组复制以匹配头数
        values = values.repeat_interleave(self.group_size, dim=1)   # 将 V 按组复制以匹配头数

        attn_scores = queries @ keys.transpose(2, 3)                # 计算注意力分数 (b, H, seq, seq)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)     # 对无效位置填充 -inf
        attn_weights = torch.softmax(                               # 归一化得到注意力权重
            attn_scores / self.head_dim**0.5, dim=-1
        )

        context = (attn_weights @ values).transpose(1, 2)           # 加权求和得到上下文 (b, seq, H, D)
        context = context.reshape(b, num_tokens, self.d_out)        # 展平头维成 (b, seq, H*D)
        return self.out_proj(context), next_cache                   # 输出投影回输入维，并返回缓存


&nbsp;
## C.5 Transformer 块


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F06_raschka.webp" width="200px">

In [18]:
class TransformerBlock(nn.Module):                          # 定义 Transformer 基本块
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(                    # 分组多头注意力层
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)                           # 前馈网络
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)       # 注意力前的归一化
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)       # 前馈前的归一化

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        shortcut = x                                         # 残差分支保存输入（注意力块）
        x = self.norm1(x)                                    # 先归一化后再注意力
        x, next_cache = self.att(                            # 注意力输出 + 更新缓存
            x, mask, cos, sin, start_pos=start_pos, cache=cache
        )  # 输出形状 [batch_size, num_tokens, emb_dim]
        x = x + shortcut                                     # 加残差，完成注意力块

        shortcut = x                                         # 残差分支保存输入（前馈块）
        x = self.norm2(x)                                    # 归一化后再送入前馈
        x = self.ff(x)                                       # 前馈网络
        x = x + shortcut                                     # 加残差，完成前馈块

        return x, next_cache                                 # 返回输出及缓存


&nbsp;
## C.6 主模型代码


In [19]:
class Qwen3Model(nn.Module):                                       # 定义 Qwen3 模型主类
    def __init__(self, cfg):
        super().__init__()

        # 主网络参数
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"],  # 词嵌入层
                                    dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(                               # 堆叠若干 TransformerBlock
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = RMSNorm(cfg["emb_dim"])                      # 输出前的归一化
        self.out_head = nn.Linear(                                     # 生成 logits 的线性头
            cfg["emb_dim"], cfg["vocab_size"],
            bias=False, dtype=cfg["dtype"]
        )

        # 预计算并缓存 RoPE 所需的 cos/sin 表
        if cfg["head_dim"] is None:                                    # 若未指定 head_dim，则按平均分配
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        cos, sin = compute_rope_params(                                # 计算 RoPE 参数
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)             # 注册 cos 表为 buffer
        self.register_buffer("sin", sin, persistent=False)             # 注册 sin 表为 buffer
        self.cfg = cfg                                                 # 保存配置
        self.current_pos = 0                                          # 记录 KV 缓存的当前位置

    def forward(self, in_idx, cache=None):
        tok_embeds = self.tok_emb(in_idx)                              # 将输入 token 转为嵌入
        x = tok_embeds                                                 # 初始化残差流

        num_tokens = x.shape[1]                                       # 当前序列长度
        if cache is not None:                                         # 若使用缓存，计算全局位置范围
            pos_start = self.current_pos
            pos_end = pos_start + num_tokens
            self.current_pos = pos_end                                 # 更新当前位置
            mask = torch.triu(                                         # 构造 (总长×总长) 的上三角遮罩
                torch.ones(
                    pos_end, pos_end, device=x.device, dtype=torch.bool
                ),
                diagonal=1
            )[pos_start:pos_end, :pos_end]                             # 截取当前步需要的子掩码
        else:
            pos_start = 0                                             # 非缓存模式从 0 开始
            mask = torch.triu(                                        # 仅基于当前序列长度构造遮罩
                torch.ones(num_tokens, num_tokens, device=x.device,
                           dtype=torch.bool),
                diagonal=1
            )
        # 预填充时遮罩形状 (num_tokens, num_tokens)
        # 解码缓存时遮罩形状 (num_tokens, 之前 + 当前 token 数)
        # 添加两个前导维度以便广播到 (batch, heads, seq, total_key_tokens)
        mask = mask[None, None, :, :]                                 # 扩展维度以便广播到注意力得分

        for i, block in enumerate(self.trf_blocks):                   # 逐层通过 TransformerBlock
            blk_cache = cache.get(i) if cache else None               # 取出该层缓存（若存在）
            x, new_blk_cache = block(x, mask, self.cos, self.sin,     # 前向计算并返回新缓存
                                     start_pos=pos_start,
                                     cache=blk_cache)
            if cache is not None:
                cache.update(i, new_blk_cache)                        # 更新缓存存储

        x = self.final_norm(x)                                        # 最终归一化
        logits = self.out_head(x.to(self.cfg["dtype"]))               # 线性映射到词表 logits
        return logits                                                 # 返回 logits

    def reset_kv_cache(self):                                         # 重置 KV 缓存位置
        self.current_pos = 0


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F07_raschka.webp" width="350px">

In [20]:
QWEN_CONFIG_06_B = {                     # 定义 Qwen3 0.6B 模型的配置字典
    "vocab_size": 151_936,               # 词表大小
    "context_length": 40_960,            # 训练时使用的上下文长度
    "emb_dim": 1024,                     # 词向量/隐藏向量维度
    "n_heads": 16,                       # 注意力头数量
    "n_layers": 28,                      # Transformer 层数
    "hidden_dim": 3072,                  # 前馈层中间维度大小
    "head_dim": 128,                     # GQA 每个头的维度
    "qk_norm": True,                     # 是否对查询/键做 RMSNorm
    "n_kv_groups": 8,                    # GQA 的 KV 分组数
    "rope_base": 1_000_000.0,            # RoPE 中 theta 的基数
    "dtype": torch.bfloat16,             # 采用 bfloat16 以节省显存
}


- Qwen3 0.6B 模型的总上下文长度支持 40,960 个 token。
- 在这 40,960 个 token 中，约 32,768 个用于模型输出（生成文本），8,192 个用于常见的提示（用户输入）。
- 模型训练时使用 32,768 token 的序列，因此该长度最能反映模型的有效工作范围。
- 32,768 个 token 已经绰绰有余；举个例子，约 4 万 token 的上下文相当于《哈利·波特》第一部篇幅的一半。


&nbsp;
## C.7 KV 缓存


In [21]:
class KVCache:                       # KV 缓存容器
    def __init__(self, n_layers):    # 传入模型层数以初始化缓存列表
        self.cache = [None] * n_layers  # 为每层预留一个缓存槽位

    def get(self, layer_idx):        # 获取指定层的缓存
        return self.cache[layer_idx]

    def update(self, layer_idx, value):  # 更新指定层的缓存内容
        self.cache[layer_idx] = value

    def get_all(self):               # 获取全部层的缓存
        return self.cache

    def reset(self):                 # 重置所有层的缓存为 None
        for i in range(len(self.cache)):
            self.cache[i] = None


&nbsp;
## C.8 分词器


In [22]:
import re                                   # 正则，用于拆分特殊 token
from tokenizers import Tokenizer             # HuggingFace Tokenizers 库
from pathlib import Path                     # 处理分词器文件路径

class Qwen3Tokenizer:
    _SPECIALS = [                            # 模型使用的特殊 token 列表
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>)") # 用正则在特殊 token 处分割文本

    def __init__(self, tokenizer_file_path="tokenizer.json",
                 apply_chat_template=False,
                 add_generation_prompt=False,
                 add_thinking=False):

        self.apply_chat_template = apply_chat_template   # 是否包裹聊天模板
        self.add_generation_prompt = add_generation_prompt  # 是否添加 assistant 提示
        self.add_thinking = add_thinking                 # 是否添加“思考”标记

        tok_path = Path(tokenizer_file_path)             # 分词器文件路径
        if not tok_path.is_file():                       # 文件不存在则报错
            raise FileNotFoundError(
                f"Tokenizer file '{tok_path}' not found. "
            )

        self._tok = Tokenizer.from_file(str(tok_path))   # 加载分词器
        self._special_to_id = {t: self._tok.token_to_id(t)   # 特殊 token 到 id 的映射
                               for t in self._SPECIALS}

        self.pad_token = "<|endoftext|>"                 # 对齐 pad token
        self.pad_token_id = self._special_to_id.get(self.pad_token)

        # 区分 Base/Reasoning：Base 用 <|endoftext|>，Reasoning 用 <|im_end|>
        fname = tok_path.name.lower()
        if "base" in fname and "reasoning" not in fname:
            self.eos_token = "<|endoftext|>"
        else:
            self.eos_token = "<|im_end|>"
        self.eos_token_id = self._special_to_id.get(self.eos_token)

    def encode(self, prompt, chat_wrapped=None):
        if chat_wrapped is None:                         # 未显式指定则用默认配置
            chat_wrapped = self.apply_chat_template

        stripped = prompt.strip()                        # 去除首尾空白
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]       # 单独的特殊 token 直接返回其 id

        if chat_wrapped:
            prompt = self._wrap_chat(prompt)             # 需要聊天模板则包裹

        ids = []                                         # 存放编码结果
        for part in filter(None, self._SPLIT_RE.split(prompt)):  # 按特殊 token 分段
            if part in self._special_to_id:              # 段落是特殊 token
                ids.append(self._special_to_id[part])
            else:                                        # 普通文本走分词器编码
                ids.extend(self._tok.encode(part).ids)
        return ids                                       # 返回 token id 列表

    def decode(self, token_ids):
        return self._tok.decode(token_ids, skip_special_tokens=False)  # 保留特殊 token 解码

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"   # 封装 user 消息
        if self.add_generation_prompt:                   # 若需 assistant 提示
            s += "<|im_start|>assistant"
            if self.add_thinking:                        # 若需“思考”换行
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"         # 默认插入 think 包裹
        return s                                         # 返回封装后的聊天文本


&nbsp;
## C.9 使用模型


In [23]:
from pathlib import Path                             # 处理文件路径
import torch                                         # PyTorch 张量/模型工具

from reasoning_from_scratch.ch02 import get_device  # noqa: F401  # 可选的设备自动选择函数
from reasoning_from_scratch.qwen3 import download_qwen3_small      # 下载 Qwen3 模型/分词器

# device = get_device()  # 可选：取消注释以自动选择设备（CUDA/MPS/CPU）
device = torch.device("cpu")                         # 本例固定使用 CPU 运行

download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")  # 下载基础版模型和分词器到本地 qwen3 目录

tokenizer_file_path = Path("qwen3") / "tokenizer-base.json"       # 分词器文件路径
model_file = Path("qwen3") / "qwen3-0.6B-base.pth"                # 模型权重文件路径

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_file_path)  # 实例化 Qwen3 分词器
model = Qwen3Model(QWEN_CONFIG_06_B)                                 # 按配置构建模型
model.load_state_dict(torch.load(model_file))                        # 加载预训练权重

model.to(device)                                                     # 将模型移动到指定设备


✓ qwen3/qwen3-0.6B-base.pth already up-to-date


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [24]:
import time  # 计时模块

from reasoning_from_scratch.ch02 import (  # 导入生成与统计工具
    generate_stats,                        # 统计生成输出信息
    generate_text_basic_cache,             # 基础生成函数（带 KV 缓存）
)

prompt = "Explain large language models in a single sentence."  # 提示词

input_token_ids_tensor = torch.tensor(       # 将提示词编码为张量
    tokenizer.encode(prompt),
    device=device
).unsqueeze(0)                               # 添加 batch 维度

start_time = time.time()                     # 记录开始时间

output_token_ids_tensor = generate_text_basic_cache(  # 生成文本（使用缓存）
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=200,                      # 生成的最大新 token 数
    eos_token_id=tokenizer.eos_token_id,     # 遇到 EOS 停止
)
end_time = time.time()                       # 记录结束时间

generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)  # 输出生成统计信息


Time: 2.92 sec
14 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.
